In [ ]:
# User:
# "Create a 30-minute workout for me. I have dumbbells,
# my knees hurt today, and I did chest yesterday."

#                     ↓

#                  Agent
#                     │
#         ┌───────────┼────────────┐
#         ↓           ↓            ↓
#  get exercises   get profile   get history
#         │           │            │
#         └───────────┼────────────┘
#                     ↓
#              reason / decide
#                     ↓
#               generate plan
#                     ↓
#               save workout

# Here the agent might decide:

# Which tools it needs

# Which tool to call first

# Whether it needs additional information

# Whether it needs to retry

# Whether it should modify the workout based on results


from typing import TypedDict, List, Any
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

import json
import os


# --------------------------------------------------
# STATE
# --------------------------------------------------

class AgentState(TypedDict):
    messages: List[BaseMessage]
    plan: List[dict]
    results: List[Any]


# --------------------------------------------------
# MOCK EXERCISE DATA
# --------------------------------------------------

MOCK_EXERCISES = [
    {
        "id": 1,
        "name": "Push Ups",
        "muscle_group": "chest",
        "equipment": "bodyweight",
        "difficulty": "beginner",
        "description": "A bodyweight exercise targeting the chest, shoulders and triceps."
    },
    {
        "id": 2,
        "name": "Bodyweight Squats",
        "muscle_group": "legs",
        "equipment": "bodyweight",
        "difficulty": "beginner",
        "description": "A lower-body exercise targeting the quads and glutes."
    },
    {
        "id": 3,
        "name": "Lunges",
        "muscle_group": "legs",
        "equipment": "bodyweight",
        "difficulty": "beginner",
        "description": "A unilateral lower-body exercise targeting the legs and glutes."
    },
    {
        "id": 4,
        "name": "Plank",
        "muscle_group": "core",
        "equipment": "bodyweight",
        "difficulty": "beginner",
        "description": "An isometric exercise targeting the core."
    },
    {
        "id": 5,
        "name": "Burpees",
        "muscle_group": "full_body",
        "equipment": "bodyweight",
        "difficulty": "intermediate",
        "description": "A full-body conditioning exercise."
    },
    {
        "id": 6,
        "name": "Mountain Climbers",
        "muscle_group": "core",
        "equipment": "bodyweight",
        "difficulty": "intermediate",
        "description": "A dynamic exercise targeting the core and cardiovascular system."
    }
]


# --------------------------------------------------
# TOOLS
# --------------------------------------------------

@tool
def fetch_exercises():
    """
    Fetch available exercises.

    Currently returns mocked exercises.
    This can later be replaced by an API/database call.
    """

    return MOCK_EXERCISES


tools = {
    "fetch_exercises": fetch_exercises
}


# --------------------------------------------------
# MODEL
# --------------------------------------------------

model = ChatOpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("OPENAI_API_KEY"),
    model=os.getenv("OPENAI_MODEL"),
    temperature=0.3,
)


# --------------------------------------------------
# PLANNER
# --------------------------------------------------

def planner_node(state: AgentState):

    response = model.invoke(
        [
            SystemMessage(
                content="""
You are a workout planning agent.

Your job is to determine what information is required
to answer the user's request.

Available tool:

- fetch_exercises()

The exercise database is currently mocked.

For workout-generation requests, always fetch the exercises
before generating the workout.

Return JSON only.

Format:

[
    {
        "task": "Fetch available exercises",
        "tool": "fetch_exercises",
        "args": {}
    }
]
"""
            )
        ]
        + state["messages"]
    )

    print("PLANNER:")
    print(response.content)

    plan = json.loads(response.content)

    return {
        **state,
        "plan": plan
    }


# --------------------------------------------------
# EXECUTOR
# --------------------------------------------------

def executor_node(state: AgentState):

    results = []

    for step in state["plan"]:

        tool_name = step["tool"]

        selected_tool = tools[tool_name]

        result = selected_tool.invoke(
            step.get("args", {})
        )

        results.append(result)

    print("EXECUTOR RESULTS:")
    print(results)

    return {
        **state,
        "results": results
    }


# --------------------------------------------------
# FINALIZER
# --------------------------------------------------

def finalizer_node(state: AgentState):

    user_question = state["messages"][0].content

    exercises = state["results"]

    response = model.invoke(
        [
            SystemMessage(
                content="""
You are a workout generation assistant.

Generate a workout based ONLY on the exercises returned
by the exercise tool.

Do not invent exercises that are not present in the
provided exercise data.

Understand the user's request and create an appropriate
workout.

For every selected exercise, provide:

- exercise name
- sets
- reps OR duration
- rest
- short instruction

Return the workout in clear JSON.

Example:

{
    "workout_name": "Beginner Full Body Workout",
    "exercises": [
        {
            "name": "Push Ups",
            "sets": 3,
            "reps": 10,
            "rest_seconds": 60
        }
    ]
}
"""
            ),
            HumanMessage(
                content=f"""
User request:
{user_question}

Available exercises:
{json.dumps(exercises, indent=2)}
"""
            )
        ]
    )

    print("FINAL RESPONSE:")
    print(response.content)

    return {
        **state,
        "messages": state["messages"] + [response]
    }


# --------------------------------------------------
# GRAPH
# --------------------------------------------------

graph = StateGraph(AgentState)

graph.add_node("planner", planner_node)
graph.add_node("executor", executor_node)
graph.add_node("finalizer", finalizer_node)

graph.add_edge(START, "planner")
graph.add_edge("planner", "executor")
graph.add_edge("executor", "finalizer")
graph.add_edge("finalizer", END)

app = graph.compile()


# --------------------------------------------------
# RUN
# --------------------------------------------------

result = app.invoke(
    {
        "messages": [
            HumanMessage(
                content="""
Create a beginner full-body workout using the
available exercises.

I want approximately 20 minutes of exercise.
"""
            )
        ],
        "plan": [],
        "results": []
    }
)

print("\nFINAL ANSWER:")
print(result["messages"][-1].content)


PLANNER:
[
    {
        "task": "Fetch available exercises",
        "tool": "fetch_exercises",
        "args": {}
    }
]
EXECUTOR RESULTS:
[[{'id': 1, 'name': 'Push Ups', 'muscle_group': 'chest', 'equipment': 'bodyweight', 'difficulty': 'beginner', 'description': 'A bodyweight exercise targeting the chest, shoulders and triceps.'}, {'id': 2, 'name': 'Bodyweight Squats', 'muscle_group': 'legs', 'equipment': 'bodyweight', 'difficulty': 'beginner', 'description': 'A lower-body exercise targeting the quads and glutes.'}, {'id': 3, 'name': 'Lunges', 'muscle_group': 'legs', 'equipment': 'bodyweight', 'difficulty': 'beginner', 'description': 'A unilateral lower-body exercise targeting the legs and glutes.'}, {'id': 4, 'name': 'Plank', 'muscle_group': 'core', 'equipment': 'bodyweight', 'difficulty': 'beginner', 'description': 'An isometric exercise targeting the core.'}, {'id': 5, 'name': 'Burpees', 'muscle_group': 'full_body', 'equipment': 'bodyweight', 'difficulty': 'intermediate', 'descr

In [1]:

# Planner → Executor → Finalizer
#    │          │           │
# "What should I do?"    "Do it"    "Present it"

# 1. Planner — "What should I do?"
# The planner looks at the user's request and decides which actions/tools are needed.

# 2. Executor — "Actually do it"
# The executor takes the planner's instructions:

# 3. Finalizer — "Now produce the answer"
# The finalizer gives all of that to the LLM and asks:


from typing import TypedDict, List, Any
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

import json
import os


# ============================================================
# STATE
# ============================================================

class AgentState(TypedDict):
    messages: List[BaseMessage]
    plan: List[dict]
    results: List[Any]


# ============================================================
# MOCK DATA
# ============================================================

MOCK_USER_PROFILE = {
    "user_id": "123",
    "goal": "weight_gain",
    "target_weight": 75,
    "current_weight": 68,
    "activity_level": "moderate",
    "experience": "intermediate",
    "training_days": 5,
    "injuries": [
        {
            "body_part": "shoulder",
            "restriction": "avoid_heavy_overhead_movements"
        }
    ]
}


MOCK_EXERCISES = [
    {
        "id": 1,
        "name": "Bench Press",
        "muscle_group": "chest",
        "equipment": "barbell",
        "difficulty": "intermediate",
        "type": "strength"
    },
    {
        "id": 2,
        "name": "Squat",
        "muscle_group": "legs",
        "equipment": "barbell",
        "difficulty": "intermediate",
        "type": "strength"
    },
    {
        "id": 3,
        "name": "Deadlift",
        "muscle_group": "back",
        "equipment": "barbell",
        "difficulty": "advanced",
        "type": "strength"
    },
    {
        "id": 4,
        "name": "Pull Ups",
        "muscle_group": "back",
        "equipment": "bodyweight",
        "difficulty": "intermediate",
        "type": "strength"
    },
    {
        "id": 5,
        "name": "Leg Press",
        "muscle_group": "legs",
        "equipment": "machine",
        "difficulty": "beginner",
        "type": "strength"
    },
    {
        "id": 6,
        "name": "Bicep Curl",
        "muscle_group": "biceps",
        "equipment": "dumbbell",
        "difficulty": "beginner",
        "type": "strength"
    },
    {
        "id": 7,
        "name": "Tricep Pushdown",
        "muscle_group": "triceps",
        "equipment": "cable",
        "difficulty": "beginner",
        "type": "strength"
    },
    {
        "id": 8,
        "name": "Lateral Raise",
        "muscle_group": "shoulders",
        "equipment": "dumbbell",
        "difficulty": "beginner",
        "type": "strength"
    }
]


# ============================================================
# TOOLS
# ============================================================

@tool
def get_user_profile(user_id: str):
    """
    Get the user's fitness profile.
    Currently returns mocked data.
    """
    return MOCK_USER_PROFILE


@tool
def fetch_exercises():
    """
    Fetch available exercises.
    Currently returns mocked exercises.
    """
    return MOCK_EXERCISES


tools = {
    "get_user_profile": get_user_profile,
    "fetch_exercises": fetch_exercises,
}


# ============================================================
# MODEL
# ============================================================

model = ChatOpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("OPENAI_API_KEY"),
    model=os.getenv("OPENAI_MODEL"),
    temperature=0.3,
)


# ============================================================
# PLANNER
# ============================================================

def planner_node(state: AgentState):

    response = model.invoke(
        [
            SystemMessage(
                content="""
You are an exercise planning agent.

Your job is to decide which tools are required
to answer the user's request.

Available tools:

{tools}

For workout-generation requests:

1. Get the user's profile.
2. Fetch the available exercises.
3. Use both pieces of information to generate the workout.

Return JSON only.

Example:

[
    {
        "task": "Get user profile",
        "tool": "get_user_profile",
        "args": {
            "user_id": "123"
        }
    },
    {
        "task": "Fetch exercises",
        "tool": "fetch_exercises",
        "args": {}
    }
]
"""
            )
        ]
        + state["messages"]
    )

    print("\n===== PLANNER =====")
    print(response.content)

    plan = json.loads(response.content)

    return {
        **state,
        "plan": plan
    }


# ============================================================
# EXECUTOR
# ============================================================

def executor_node(state: AgentState):

    results = []

    for step in state["plan"]:

        tool_name = step["tool"]

        selected_tool = tools[tool_name]

        result = selected_tool.invoke(
            step.get("args", {})
        )

        results.append({
            "task": step["task"],
            "result": result
        })

    print("\n===== EXECUTOR =====")
    print(json.dumps(results, indent=2))

    return {
        **state,
        "results": results
    }


# ============================================================
# FINALIZER
# ============================================================

def finalizer_node(state: AgentState):

    user_request = state["messages"][0].content

    response = model.invoke(
        [
            SystemMessage(
                content="""
You are an expert workout generation assistant.

Generate a workout based on:

1. The user's request.
2. The user's profile.
3. The available exercises.

Important rules:

- Only use exercises returned by the exercise tool.
- Respect the user's training experience.
- Respect the requested number of training days.
- Consider the user's goal.
- Consider the user's injuries and limitations.
- Do not invent exercises.
- Generate a structured workout split.

Return JSON only.

Example format:

{
    "goal": "weight_gain",
    "training_days": 5,
    "split": [
        {
            "day": 1,
            "focus": "Chest and Triceps",
            "exercises": [
                {
                    "name": "Bench Press",
                    "sets": 4,
                    "reps": 8,
                    "rest_seconds": 120
                }
            ]
        }
    ]
}
"""
            ),
            HumanMessage(
                content=f"""
User request:

{user_request}

Information collected by the agent:

{json.dumps(state["results"], indent=2)}
"""
            )
        ]
    )

    print("\n===== FINALIZER =====")
    print(response.content)

    return {
        **state,
        "messages": state["messages"] + [response]
    }


# ============================================================
# GRAPH
# ============================================================

graph = StateGraph(AgentState)

graph.add_node("planner", planner_node)
graph.add_node("executor", executor_node)
graph.add_node("finalizer", finalizer_node)

graph.add_edge(START, "planner")
graph.add_edge("planner", "executor")
graph.add_edge("executor", "finalizer")
graph.add_edge("finalizer", END)

app = graph.compile()


# ============================================================
# RUN
# ============================================================

result = app.invoke(
    {
        "messages": [
            HumanMessage(
                content="""
Create a 5-day workout plan for me.

My goal is to gain weight.
I want a strength-focused workout.
"""
            )
        ],
        "plan": [],
        "results": []
    }
)


print("\n==============================")
print("FINAL WORKOUT")
print("==============================")

print(result["messages"][-1].content)

/home/scientist-ubuntu/projects/all_venv_ubuntu/vvenv_extraction/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



===== PLANNER =====
[
    {
        "task": "Get user profile",
        "tool": "get_user_profile",
        "args": {
            "user_id": "123"
        }
    },
    {
        "task": "Fetch exercises",
        "tool": "fetch_exercises",
        "args": {
            "focus": "strength",
            "goal": "gain weight"
        }
    }
]

===== EXECUTOR =====
[
  {
    "task": "Get user profile",
    "result": {
      "user_id": "123",
      "goal": "weight_gain",
      "target_weight": 75,
      "current_weight": 68,
      "activity_level": "moderate",
      "experience": "intermediate",
      "training_days": 5,
      "injuries": [
        {
          "body_part": "shoulder",
          "restriction": "avoid_heavy_overhead_movements"
        }
      ]
    }
  },
  {
    "task": "Fetch exercises",
    "result": [
      {
        "id": 1,
        "name": "Bench Press",
        "muscle_group": "chest",
        "equipment": "barbell",
        "difficulty": "intermediate",
        "typ

In [4]:

# ============================================================
# MOCK DATA
# ============================================================

MOCK_USER_PROFILE = {
    "user_id": "123",
    "goal": "weight_gain",
    "target_weight": 75,
    "current_weight": 68,
    "activity_level": "moderate",
    "experience": "intermediate",
    "training_days": 5,
    "injuries": [
        {
            "body_part": "shoulder",
            "restriction": "avoid_heavy_overhead_movements"
        }
    ]
}


MOCK_EXERCISES = [
    {
        "id": 1,
        "name": "Bench Press",
        "muscle_group": "chest",
        "equipment": "barbell",
        "difficulty": "intermediate",
        "type": "strength"
    },
    {
        "id": 2,
        "name": "Squat",
        "muscle_group": "legs",
        "equipment": "barbell",
        "difficulty": "intermediate",
        "type": "strength"
    },
    {
        "id": 3,
        "name": "Deadlift",
        "muscle_group": "back",
        "equipment": "barbell",
        "difficulty": "advanced",
        "type": "strength"
    },
    {
        "id": 4,
        "name": "Pull Ups",
        "muscle_group": "back",
        "equipment": "bodyweight",
        "difficulty": "intermediate",
        "type": "strength"
    },
    {
        "id": 5,
        "name": "Leg Press",
        "muscle_group": "legs",
        "equipment": "machine",
        "difficulty": "beginner",
        "type": "strength"
    },
    {
        "id": 6,
        "name": "Bicep Curl",
        "muscle_group": "biceps",
        "equipment": "dumbbell",
        "difficulty": "beginner",
        "type": "strength"
    },
    {
        "id": 7,
        "name": "Tricep Pushdown",
        "muscle_group": "triceps",
        "equipment": "cable",
        "difficulty": "beginner",
        "type": "strength"
    },
    {
        "id": 8,
        "name": "Lateral Raise",
        "muscle_group": "shoulders",
        "equipment": "dumbbell",
        "difficulty": "beginner",
        "type": "strength"
    }
]


In [ ]:
########## Validations layer

from typing import TypedDict, List, Any, Literal
from pydantic import BaseModel, Field
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

import json
import os



# ============================================================
# STATE
# ============================================================

class AgentState(TypedDict):
    messages: List[BaseMessage]
    plan: List[dict]
    results: List[dict]
    final_workout: dict


# ============================================================
# PLANNER SCHEMAS
# ============================================================

class PlanStep(BaseModel):
    task: str
    tool: Literal[
        "get_user_profile",
        "fetch_exercises"
    ]
    args: dict = Field(default_factory=dict)


class WorkoutPlan(BaseModel):
    steps: List[PlanStep]


# ============================================================
# FINALIZER SCHEMAS
# ============================================================

class ExercisePlan(BaseModel):
    name: str
    sets: int
    reps: int
    rest_seconds: int


class TrainingDay(BaseModel):
    day: int
    focus: str
    exercises: List[ExercisePlan]


class FinalWorkout(BaseModel):
    goal: str
    training_days: int
    split: List[TrainingDay]


# ============================================================
# TOOLS
# ============================================================

@tool
def get_user_profile(user_id: str):
    """
    Get the user's fitness profile.
    """
    return MOCK_USER_PROFILE


@tool
def fetch_exercises():
    """
    Fetch all available exercises.
    """
    return MOCK_EXERCISES


TOOLS = {
    "get_user_profile": get_user_profile,
    "fetch_exercises": fetch_exercises,
}


# ============================================================
# MODEL
# ============================================================

model = ChatOpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("OPENAI_API_KEY"),
    model=os.getenv("OPENAI_MODEL"),
    temperature=0.3,
)


# ============================================================
# PLANNER
# ============================================================

def planner_node(state: AgentState):

    planner_model = model.with_structured_output(WorkoutPlan)

    response = planner_model.invoke(
        [
            SystemMessage(
                content="""
You are the planning component of a fitness agent.

Your job is ONLY to decide which tools are required.

Do NOT generate a workout.

For a workout-generation request:

1. Get the user's profile.
2. Fetch the available exercises.

Available tools:

- get_user_profile(user_id: str)
- fetch_exercises()

IMPORTANT:

- Use only the tools listed above.
- Do not invent tools.
- Do not add arguments that a tool does not accept.
- get_user_profile requires user_id.
- fetch_exercises takes no arguments.
"""
            ),
            *state["messages"]
        ]
    )

    plan = response.model_dump()["steps"]

    print("\n===== PLANNER =====")
    print(json.dumps(plan, indent=2))

    return {
        **state,
        "plan": plan
    }


# ============================================================
# EXECUTOR
# ============================================================

def executor_node(state: AgentState):

    results = []

    for step in state["plan"]:

        tool_name = step["tool"]

        if tool_name not in TOOLS:
            raise ValueError(
                f"Planner requested unknown tool: {tool_name}"
            )

        selected_tool = TOOLS[tool_name]

        args = step.get("args", {})

        try:
            result = selected_tool.invoke(args)
        except Exception as e:
            raise RuntimeError(
                f"Tool '{tool_name}' failed: {e}"
            ) from e

        results.append(
            {
                "task": step["task"],
                "tool": tool_name,
                "result": result
            }
        )

    print("\n===== EXECUTOR =====")
    print(json.dumps(results, indent=2))

    return {
        **state,
        "results": results
    }


# ============================================================
# DATA EXTRACTION HELPERS
# ============================================================

def extract_profile(results: List[dict]) -> dict:

    for item in results:
        if item["tool"] == "get_user_profile":
            return item["result"]

    raise ValueError("User profile was not collected.")


def extract_exercises(results: List[dict]) -> List[dict]:

    for item in results:
        if item["tool"] == "fetch_exercises":
            return item["result"]

    raise ValueError("Exercises were not collected.")


# ============================================================
# VALIDATION
# ============================================================

def validate_workout(
    workout: dict,
    profile: dict,
    exercises: List[dict]
):

    # --------------------------------------------------------
    # Validate training days
    # --------------------------------------------------------

    expected_days = profile["training_days"]

    if workout["training_days"] != expected_days:
        raise ValueError(
            f"Expected {expected_days} training days, "
            f"got {workout['training_days']}"
        )

    if len(workout["split"]) != expected_days:
        raise ValueError(
            f"Expected {expected_days} workout days, "
            f"got {len(workout['split'])}"
        )

    # --------------------------------------------------------
    # Build allowed exercise set
    # --------------------------------------------------------

    allowed_exercises = {
        exercise["name"]
        for exercise in exercises
    }

    # --------------------------------------------------------
    # Validate every exercise
    # --------------------------------------------------------

    for day in workout["split"]:

        for exercise in day["exercises"]:

            name = exercise["name"]

            if name not in allowed_exercises:
                raise ValueError(
                    f"Invalid exercise generated: '{name}'. "
                    f"It was not returned by fetch_exercises()."
                )

            if exercise["sets"] <= 0:
                raise ValueError(
                    f"Invalid sets for {name}"
                )

            if exercise["reps"] <= 0:
                raise ValueError(
                    f"Invalid reps for {name}"
                )

            if exercise["rest_seconds"] < 0:
                raise ValueError(
                    f"Invalid rest time for {name}"
                )

    # --------------------------------------------------------
    # Validate day numbers
    # --------------------------------------------------------

    day_numbers = {
        day["day"]
        for day in workout["split"]
    }

    expected_day_numbers = set(range(1, expected_days + 1))

    if day_numbers != expected_day_numbers:
        raise ValueError(
            f"Invalid workout day numbers: {day_numbers}"
        )

    return True


# ============================================================
# FINALIZER
# ============================================================

def finalizer_node(state: AgentState):

    user_request = state["messages"][0].content

    profile = extract_profile(state["results"])
    exercises = extract_exercises(state["results"])

    finalizer_model = model.with_structured_output(FinalWorkout)

    response = finalizer_model.invoke(
        [
            SystemMessage(
                content="""
You are the final workout-generation component.

Generate a workout using ONLY the information provided by
the executor.

STRICT RULES:

1. Only use exercises from the available exercise list.
2. NEVER invent an exercise.
3. Respect the user's training experience.
4. Respect the requested number of training days.
5. Consider the user's goal.
6. Consider injuries and restrictions.
7. Do not create exercises that are not in the database.
8. Do not add nutritional recommendations.
9. Return only the structured workout.

If the exercise database does not contain a suitable exercise
for a particular muscle group, do not invent one.

The application will validate your output.
"""
            ),
            HumanMessage(
                content=f"""
USER REQUEST:

{user_request}

USER PROFILE:

{json.dumps(profile, indent=2)}

AVAILABLE EXERCISES:

{json.dumps(exercises, indent=2)}
"""
            )
        ]
    )

    workout = response.model_dump()

    # --------------------------------------------------------
    # IMPORTANT:
    # Validate LLM output BEFORE returning it.
    # --------------------------------------------------------

    validate_workout(
        workout=workout,
        profile=profile,
        exercises=exercises
    )

    print("\n===== FINALIZER =====")
    print(json.dumps(workout, indent=2))

    return {
        **state,
        "final_workout": workout,
        "messages": state["messages"] + [
            HumanMessage(
                content=json.dumps(workout, indent=2)
            )
        ]
    }


# ============================================================
# GRAPH
# ============================================================

graph = StateGraph(AgentState)

graph.add_node("planner", planner_node)
graph.add_node("executor", executor_node)
graph.add_node("finalizer", finalizer_node)

graph.add_edge(START, "planner")
graph.add_edge("planner", "executor")
graph.add_edge("executor", "finalizer")
graph.add_edge("finalizer", END)

app = graph.compile()


# ============================================================
# RUN
# ============================================================

result = app.invoke(
    {
        "messages": [
            HumanMessage(
                content="""
Create a 5-day workout plan for me.

My goal is to gain weight.
I want a strength-focused workout.
"""
            )
        ],
        "plan": [],
        "results": [],
        "final_workout": {}
    }
)


# ============================================================
# FINAL OUTPUT
# ============================================================

print("\n==============================")
print("FINAL WORKOUT")
print("==============================")

print(
    json.dumps(
        result["final_workout"],
        indent=2
    )
)



===== PLANNER =====
[
  {
    "task": "get_user_profile",
    "tool": "get_user_profile",
    "args": {
      "user_id": "user123"
    }
  },
  {
    "task": "fetch_exercises",
    "tool": "fetch_exercises",
    "args": {}
  }
]

===== EXECUTOR =====
[
  {
    "task": "get_user_profile",
    "tool": "get_user_profile",
    "result": {
      "user_id": "123",
      "goal": "weight_gain",
      "target_weight": 75,
      "current_weight": 68,
      "activity_level": "moderate",
      "experience": "intermediate",
      "training_days": 5,
      "injuries": [
        {
          "body_part": "shoulder",
          "restriction": "avoid_heavy_overhead_movements"
        }
      ]
    }
  },
  {
    "task": "fetch_exercises",
    "tool": "fetch_exercises",
    "result": [
      {
        "id": 1,
        "name": "Bench Press",
        "muscle_group": "chest",
        "equipment": "barbell",
        "difficulty": "intermediate",
        "type": "strength"
      },
      {
        "id": 2,
  

In [6]:
#### New Validation layer

        #              USER REQUEST
        #                   │
        #                   ▼
        #             ┌──────────┐
        #             │ PLANNER  │
        #             └────┬─────┘
        #                  │
        #           structured plan
        #                  │
        #                  ▼
        #             ┌──────────┐
        #             │ EXECUTOR │
        #             └────┬─────┘
        #                  │
        #        ┌─────────┴─────────┐
        #        ▼                   ▼
        # User Profile         Exercise DB
        #        │                   │
        #        └─────────┬─────────┘
        #                  ▼
        #         ┌─────────────────┐
        #         │ CONSTRAINT      │
        #         │ ENGINE          │
        #         └────────┬────────┘
        #                  │
        #           valid context
        #                  │
        #                  ▼
        #            ┌───────────┐
        #            │ FINALIZER │
        #            └─────┬─────┘
        #                  │
        #             structured
        #               workout
        #                  │
        #                  ▼
        #            ┌───────────┐
        #            │ VALIDATOR │
        #            └─────┬─────┘
        #                  │
        #                  ▼
        #             FINAL JSON


from typing import TypedDict, List, Any, Literal
from pydantic import BaseModel, Field
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

import json
import os


# ============================================================
# MOCK DATA
# ============================================================
MOCK_EXERCISES = [

    # -------------------------
    # VALID
    # -------------------------

    {
        "id": 1,
        "name": "Bench Press",
        "muscle_group": "chest",
        "equipment": "barbell",
        "difficulty": "intermediate",
        "type": "strength",

        "restrictions": []
    },

    {
        "id": 2,
        "name": "Squat",
        "muscle_group": "legs",
        "equipment": "barbell",
        "difficulty": "intermediate",
        "type": "strength",

        "restrictions": []
    },

    # -------------------------
    # SHOULDER PROBLEM
    # -------------------------

    {
        "id": 9,
        "name": "Heavy Overhead Press",
        "muscle_group": "shoulders",
        "equipment": "barbell",
        "difficulty": "advanced",
        "type": "strength",

        "restrictions": [
            "shoulder_injury"
        ]
    },

    # -------------------------
    # KNEE PROBLEM
    # -------------------------

    {
        "id": 10,
        "name": "Heavy Barbell Squat",
        "muscle_group": "legs",
        "equipment": "barbell",
        "difficulty": "advanced",
        "type": "strength",

        "restrictions": [
            "knee_injury"
        ]
    },

    # -------------------------
    # BACK PROBLEM
    # -------------------------

    {
        "id": 11,
        "name": "Heavy Deadlift",
        "muscle_group": "back",
        "equipment": "barbell",
        "difficulty": "advanced",
        "type": "strength",

        "restrictions": [
            "lower_back_injury"
        ]
    }
]

MOCK_USER_PROFILE = {
    "user_id": "123",

    "goal": "weight_gain",

    "target_weight": 75,

    "current_weight": 68,

    "activity_level": "moderate",

    "experience": "intermediate",

    "training_days": 5,

    "injuries": [
        {
            "body_part": "shoulder",
            "restriction": "avoid_heavy_overhead_movements"
        },
        {
            "body_part": "knee",
            "restriction": "avoid_heavy_knee_loading"
        }
    ]
}



# ============================================================
# STATE
# ============================================================

class AgentState(TypedDict):
    messages: List[BaseMessage]

    # Application-owned identity.
    user_id: str

    # Planner output.
    plan: List[dict]

    # Executor output.
    results: List[dict]

    # Constraint engine output.
    constrained_context: dict

    # Finalizer output.
    final_workout: dict

    # Validator output.
    validation_errors: List[str]

    # Number of finalizer attempts.
    retry_count: int


# ============================================================
# PLANNER SCHEMAS
# ============================================================

class PlanStep(BaseModel):
    task: str
    tool: Literal[
        "get_user_profile",
        "fetch_exercises"
    ]
    args: dict = Field(default_factory=dict)


class PlannerOutput(BaseModel):
    steps: List[PlanStep]


# ============================================================
# FINALIZER SCHEMAS
# ============================================================

class ExercisePlan(BaseModel):
    name: str
    sets: int
    reps: int
    rest_seconds: int


class TrainingDay(BaseModel):
    day: int
    focus: str
    exercises: List[ExercisePlan]


class FinalWorkout(BaseModel):
    goal: str
    training_days: int
    split: List[TrainingDay]


# ============================================================
# TOOLS
# ============================================================

@tool
def get_user_profile(user_id: str):
    """
    Retrieve a user's fitness profile.
    """

    if user_id != MOCK_USER_PROFILE["user_id"]:
        raise ValueError(
            f"User '{user_id}' does not exist."
        )

    return MOCK_USER_PROFILE


@tool
def fetch_exercises():
    """
    Retrieve all available exercises.
    """

    return MOCK_EXERCISES


TOOLS = {
    "get_user_profile": get_user_profile,
    "fetch_exercises": fetch_exercises,
}


# ============================================================
# MODEL
# ============================================================

model = ChatOpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("OPENAI_API_KEY"),
    model=os.getenv("OPENAI_MODEL"),
    temperature=0.3,
)


# ============================================================
# 1. PLANNER
# ============================================================

def planner_node(state: AgentState):

    planner_model = model.with_structured_output(
        PlannerOutput
    )

    user_id = state["user_id"]

    response = planner_model.invoke(
        [
            SystemMessage(
                content=f"""
You are the PLANNER in a workout-generation agent.

Your responsibility is ONLY to decide which tools need
to be called.

Do NOT generate a workout.

Available tools:

1. get_user_profile(user_id: str)

2. fetch_exercises()

Rules:

- Use only the tools listed above.
- get_user_profile requires a user_id.
- fetch_exercises takes no arguments.
- The application has already provided the user ID.
- Always use this exact user ID:

{user_id}

For workout-generation requests, normally collect:
- user profile
- available exercises
"""
            ),
            *state["messages"]
        ]
    )

    plan = [
        step.model_dump()
        for step in response.steps
    ]

    print("\n===== PLANNER =====")
    print(json.dumps(plan, indent=2))

    return {
        **state,
        "plan": plan
    }


# ============================================================
# 2. EXECUTOR
# ============================================================

def executor_node(state: AgentState):

    results = []

    for step in state["plan"]:

        tool_name = step["tool"]

        # ----------------------------------------------------
        # Never allow unknown tools.
        # ----------------------------------------------------

        if tool_name not in TOOLS:
            raise ValueError(
                f"Unknown tool requested: {tool_name}"
            )

        selected_tool = TOOLS[tool_name]

        # ----------------------------------------------------
        # Execute tool.
        # ----------------------------------------------------

        try:
            result = selected_tool.invoke(
                step.get("args", {})
            )

        except Exception as e:

            raise RuntimeError(
                f"Tool '{tool_name}' failed: {e}"
            ) from e

        results.append(
            {
                "task": step["task"],
                "tool": tool_name,
                "result": result
            }
        )

    print("\n===== EXECUTOR =====")
    print(
        json.dumps(
            results,
            indent=2
        )
    )

    return {
        **state,
        "results": results
    }


# ============================================================
# HELPERS
# ============================================================

def get_profile_from_results(results):

    for item in results:

        if item["tool"] == "get_user_profile":
            return item["result"]

    raise ValueError(
        "User profile was not returned by executor."
    )


def get_exercises_from_results(results):

    for item in results:

        if item["tool"] == "fetch_exercises":
            return item["result"]

    raise ValueError(
        "Exercise list was not returned by executor."
    )


# ============================================================
# 3. CONSTRAINT ENGINE
# ============================================================

def constraint_node(state: AgentState):

    profile = get_profile_from_results(
        state["results"]
    )

    exercises = get_exercises_from_results(
        state["results"]
    )

    allowed_exercises = []

    restricted_exercises = []

    # --------------------------------------------------------
    # Determine whether shoulder restriction exists.
    # --------------------------------------------------------

    shoulder_restriction = False

    for injury in profile.get("injuries", []):

        if (
            injury["body_part"] == "shoulder"
            and injury["restriction"]
            == "avoid_heavy_overhead_movements"
        ):
            shoulder_restriction = True

    # --------------------------------------------------------
    # Apply deterministic constraints.
    # --------------------------------------------------------

    for exercise in exercises:

        # Current mock rule:
        #
        # If user has a shoulder restriction, exclude
        # exercises explicitly marked as overhead.
        #
        # Notice that Lateral Raise remains available because
        # this mock database marks it as overhead=False.
        #
        # Your real application can have more sophisticated
        # exercise metadata/rules.

        if (
            shoulder_restriction
            and exercise.get("overhead") is True
        ):
            restricted_exercises.append(
                {
                    "name": exercise["name"],
                    "reason": "shoulder restriction"
                }
            )

            continue

        allowed_exercises.append(exercise)

    constrained_context = {
        "profile": profile,

        "allowed_exercises": allowed_exercises,

        "restricted_exercises": restricted_exercises,

        "constraints": [
            injury["restriction"]
            for injury in profile.get("injuries", [])
        ]
    }

    print("\n===== CONSTRAINT ENGINE =====")
    print(
        json.dumps(
            constrained_context,
            indent=2
        )
    )

    return {
        **state,
        "constrained_context": constrained_context
    }


# ============================================================
# 4. FINALIZER
# ============================================================

def finalizer_node(state: AgentState):

    context = state["constrained_context"]

    profile = context["profile"]

    allowed_exercises = context["allowed_exercises"]

    validation_errors = state.get(
        "validation_errors",
        []
    )

    # --------------------------------------------------------
    # Build retry feedback.
    # --------------------------------------------------------

    retry_feedback = ""

    if validation_errors:

        retry_feedback = f"""
The previous generated workout failed validation.

Errors:

{json.dumps(validation_errors, indent=2)}

You MUST correct these errors.

Do not repeat them.
"""

    # --------------------------------------------------------
    # Structured output.
    # --------------------------------------------------------

    finalizer_model = model.with_structured_output(
        FinalWorkout
    )

    response = finalizer_model.invoke(
        [
            SystemMessage(
                content=f"""
You are the FINALIZER in a workout-generation system.

Your job is to generate the final structured workout.

USER PROFILE:

{json.dumps(profile, indent=2)}

ALLOWED EXERCISES:

{json.dumps(allowed_exercises, indent=2)}

IMPORTANT RULES:

1. ONLY use exercises from ALLOWED EXERCISES.

2. NEVER invent exercises.

3. The workout must contain exactly:

{profile["training_days"]}

training days.

4. Respect the user's experience level:

{profile["experience"]}

5. Consider the user's goal:

{profile["goal"]}

6. Respect all restrictions:

{json.dumps(context["constraints"], indent=2)}

7. Do not invent exercises to fill missing muscle groups.

8. If an exercise is unavailable, simply construct the
   workout using the available exercises.

9. Do not add nutrition advice.

10. Return only the structured workout.

{retry_feedback}
"""
            ),
            HumanMessage(
                content=state["messages"][0].content
            )
        ]
    )

    workout = response.model_dump()

    print("\n===== FINALIZER =====")
    print(
        json.dumps(
            workout,
            indent=2
        )
    )

    return {
        **state,
        "final_workout": workout,
        "retry_count": state["retry_count"] + 1
    }


# ============================================================
# 5. VALIDATOR
# ============================================================

def validator_node(state: AgentState):

    workout = state["final_workout"]

    profile = state[
        "constrained_context"
    ]["profile"]

    allowed_exercises = state[
        "constrained_context"
    ]["allowed_exercises"]

    errors = []

    # --------------------------------------------------------
    # Allowed exercise names.
    # --------------------------------------------------------

    allowed_names = {
        exercise["name"]
        for exercise in allowed_exercises
    }

    # --------------------------------------------------------
    # Training days.
    # --------------------------------------------------------

    expected_days = profile["training_days"]

    if workout["training_days"] != expected_days:

        errors.append(
            f"Expected training_days={expected_days}, "
            f"got {workout['training_days']}"
        )

    if len(workout["split"]) != expected_days:

        errors.append(
            f"Expected {expected_days} workout days, "
            f"got {len(workout['split'])}"
        )

    # --------------------------------------------------------
    # Day numbers.
    # --------------------------------------------------------

    actual_days = {
        day["day"]
        for day in workout["split"]
    }

    expected_day_numbers = set(
        range(1, expected_days + 1)
    )

    if actual_days != expected_day_numbers:

        errors.append(
            f"Invalid day numbers: {actual_days}"
        )

    # --------------------------------------------------------
    # Exercise validation.
    # --------------------------------------------------------

    for day in workout["split"]:

        for exercise in day["exercises"]:

            name = exercise["name"]

            # Unknown exercise
            if name not in allowed_names:

                errors.append(
                    f"Exercise '{name}' is not allowed."
                )

            # Invalid sets
            if exercise["sets"] <= 0:

                errors.append(
                    f"{name}: sets must be > 0."
                )

            # Invalid reps
            if exercise["reps"] <= 0:

                errors.append(
                    f"{name}: reps must be > 0."
                )

            # Invalid rest
            if exercise["rest_seconds"] < 0:

                errors.append(
                    f"{name}: rest_seconds cannot be negative."
                )

    # --------------------------------------------------------
    # Print validation result.
    # --------------------------------------------------------

    if errors:

        print("\n===== VALIDATOR =====")
        print("INVALID")

        for error in errors:
            print(f"- {error}")

    else:

        print("\n===== VALIDATOR =====")
        print("VALID")

    return {
        **state,
        "validation_errors": errors
    }


# ============================================================
# CONDITIONAL ROUTING
# ============================================================

def validation_router(state: AgentState):

    errors = state.get(
        "validation_errors",
        []
    )

    # --------------------------------------------------------
    # Valid → finish.
    # --------------------------------------------------------

    if not errors:
        return "end"

    # --------------------------------------------------------
    # Invalid → retry finalizer.
    #
    # Maximum 2 finalizer attempts.
    # --------------------------------------------------------

    if state["retry_count"] >= 2:

        print(
            "\nMaximum finalizer retries reached."
        )

        return "end"

    print(
        "\nValidation failed. "
        "Sending feedback back to finalizer..."
    )

    return "retry"


# ============================================================
# GRAPH
# ============================================================

graph = StateGraph(AgentState)

graph.add_node(
    "planner",
    planner_node
)

graph.add_node(
    "executor",
    executor_node
)

graph.add_node(
    "constraints",
    constraint_node
)

graph.add_node(
    "finalizer",
    finalizer_node
)

graph.add_node(
    "validator",
    validator_node
)


# ============================================================
# EDGES
# ============================================================

graph.add_edge(
    START,
    "planner"
)

graph.add_edge(
    "planner",
    "executor"
)

graph.add_edge(
    "executor",
    "constraints"
)

graph.add_edge(
    "constraints",
    "finalizer"
)

graph.add_edge(
    "finalizer",
    "validator"
)


graph.add_conditional_edges(
    "validator",
    validation_router,
    {
        "retry": "finalizer",
        "end": END
    }
)


# ============================================================
# COMPILE
# ============================================================

app = graph.compile()


# ============================================================
# RUN
# ============================================================

user_request = """
Create a 5-day workout plan for me.

My goal is to gain weight.
I want a strength-focused workout.
"""


result = app.invoke(
    {
        "messages": [
            HumanMessage(
                content=user_request
            )
        ],

        # ----------------------------------------------------
        # IMPORTANT:
        # User ID comes from application state.
        # The LLM does NOT invent it.
        # ----------------------------------------------------

        "user_id": "123",

        "plan": [],

        "results": [],

        "constrained_context": {},

        "final_workout": {},

        "validation_errors": [],

        "retry_count": 0
    }
)


# ============================================================
# FINAL OUTPUT
# ============================================================

print("\n")
print("=" * 60)
print("FINAL WORKOUT")
print("=" * 60)

print(
    json.dumps(
        result["final_workout"],
        indent=2
    )
)

print("\n")
print("=" * 60)
print("VALIDATION ERRORS")
print("=" * 60)

print(
    json.dumps(
        result["validation_errors"],
        indent=2
    )
)



===== PLANNER =====
[
  {
    "task": "get_user_profile",
    "tool": "get_user_profile",
    "args": {
      "user_id": "123"
    }
  },
  {
    "task": "fetch_exercises",
    "tool": "fetch_exercises",
    "args": {}
  }
]

===== EXECUTOR =====
[
  {
    "task": "get_user_profile",
    "tool": "get_user_profile",
    "result": {
      "user_id": "123",
      "goal": "weight_gain",
      "target_weight": 75,
      "current_weight": 68,
      "activity_level": "moderate",
      "experience": "intermediate",
      "training_days": 5,
      "injuries": [
        {
          "body_part": "shoulder",
          "restriction": "avoid_heavy_overhead_movements"
        },
        {
          "body_part": "knee",
          "restriction": "avoid_heavy_knee_loading"
        }
      ]
    }
  },
  {
    "task": "fetch_exercises",
    "tool": "fetch_exercises",
    "result": [
      {
        "id": 1,
        "name": "Bench Press",
        "muscle_group": "chest",
        "equipment": "barbell",
 